# 用 AI × Colab 分析 Titanic 乘客資料

大數據分析與實踐 · R 寶的 AI × Colab 實戰。每一步上面的引用框是當時給 AI 的 prompt，下面是 AI 寫好、我們檢查過的程式碼。

教材網頁：https://chenyc0901.github.io/R_datastructure_learning/colab.html

**🤖 給 AI 的 prompt**

> 你是一位資料分析助教，熟悉 Python 的 pandas、seaborn、scipy 和 statsmodels。
> 我是醫技系的大學生，正在 Google Colab 上分析 Titanic 乘客資料（891 位乘客，欄位有 PassengerId、Survived、Pclass、Name、Sex、Age、SibSp、Parch、Ticket、Fare、Cabin、Embarked）。
> 接下來我會一步一步請你寫程式，請遵守：
> 1. 每次只做我要求的那一步，不要一次做完全部。
> 2. 程式碼加上繁體中文註解。
> 3. 圖上的文字一律用英文（Colab 預設沒有中文字型）。
> 4. 不要自己編造數字；結果以我執行後貼回給你的輸出為準。

## 步驟 1　讀取資料

**🤖 給 AI 的 prompt**

> 請寫 Python 程式：從下面的網址讀取 CSV，存到變數 df，印出它有幾列幾欄，再顯示前 5 列。
> https://chenyc0901.github.io/R_datastructure_learning/data/titanic.csv

In [ ]:
import pandas as pd                 # 表格資料
import matplotlib.pyplot as plt     # 畫圖的底層
import seaborn as sns               # 比較漂亮、好寫的統計圖

url = "https://chenyc0901.github.io/R_datastructure_learning/data/titanic.csv"
df = pd.read_csv(url)               # 讀 CSV，存成 DataFrame
print(df.shape)                     # (列數, 欄數)
df.head()                           # 前 5 列

## 步驟 2　認識資料

**🤖 給 AI 的 prompt**

> 請用 info() 和 describe() 檢查 df：每一欄的資料型態、非缺值的數量，以及數值欄位的摘要統計。
> 執行結果我貼在下面，請用白話說明每一欄代表什麼，以及你看到哪些需要注意的地方。
> （貼上輸出）

In [ ]:
df.info()                           # 每一欄的型態與非缺值數量

In [ ]:
df.describe().round(2)              # 數值欄位的摘要統計

## 步驟 3　缺值與清理

**🤖 給 AI 的 prompt**

> 請算出 df 每一欄有幾個缺值、佔全部的百分比，整理成一張表。

In [ ]:
na = df.isna().sum()                              # 每一欄的缺值數
pd.DataFrame({"n_missing": na,
              "percent": (na / len(df) * 100).round(1)})   # 佔全部的 %

**🤖 給 AI 的 prompt**

> Cabin 缺 687 筆、Age 缺 177 筆、Embarked 缺 2 筆。
> 請先不要寫程式，列出每一欄可以怎麼處理（刪掉欄位、補值、保留缺值），以及各自的優缺點。

**🤖 給 AI 的 prompt**

> 好，請照下面的決定寫程式，結果存成新的 DataFrame clean，不要改動 df：
> 1. 刪掉 Cabin 和 Ticket 兩欄。
> 2. Embarked 用出現最多次的值補。
> 3. Age 保留缺值，不要補。
> 4. 新增 FamilySize = SibSp + Parch + 1。
> 5. 新增 AgeGroup，把 Age 分成 0-12、13-18、19-40、41-60、61-80。
> 6. 新增 Outcome，把 Survived 的 0 / 1 換成 "Died" / "Survived"。
> 最後檢查 clean 每一欄還有幾個缺值。

In [ ]:
clean = df.drop(columns=["Cabin", "Ticket"])     # drop 會回傳新的表，df 不會被改到
clean["Embarked"] = clean["Embarked"].fillna(clean["Embarked"].mode()[0])   # 補上最常見的港口
clean["FamilySize"] = clean["SibSp"] + clean["Parch"] + 1                   # 兄弟姊妹配偶 + 父母子女 + 自己
clean["AgeGroup"] = pd.cut(clean["Age"],
                           bins=[0, 12, 18, 40, 60, 80],                    # (0,12]、(12,18]……
                           labels=["0-12", "13-18", "19-40", "41-60", "61-80"])
clean["Outcome"] = clean["Survived"].map({0: "Died", 1: "Survived"})        # 0/1 換成文字，畫圖比較好讀
clean.isna().sum()

## 步驟 4　探索：誰比較容易活下來？

**🤖 給 AI 的 prompt**

> 請用 clean 算出：
> 1. 整體存活率。
> 2. 依性別（Sex）、艙等（Pclass）各自分組的人數、存活人數與存活率。
> 3. 艙等 × 性別交叉的存活率表。
> 4. 依年齡組（AgeGroup）、家庭人數（FamilySize）分組的人數與存活率。
> 每一個結果分開放在不同的儲存格，存活率取到小數 3 位。

In [ ]:
print("總人數：", len(clean))
print("存活率：", round(clean["Survived"].mean() * 100, 1), "%")   # 0/1 的平均 = 1 的比例
clean["Outcome"].value_counts()

In [ ]:
clean.groupby("Sex")["Survived"].agg(n="count", survived="sum", rate="mean").round(3)

In [ ]:
clean.groupby("Pclass")["Survived"].agg(n="count", survived="sum", rate="mean").round(3)

In [ ]:
pd.crosstab(clean["Pclass"], clean["Sex"],
            values=clean["Survived"], aggfunc="mean").round(3)   # 每一格是存活率

In [ ]:
clean.groupby("AgeGroup", observed=True)["Survived"].agg(n="count", rate="mean").round(3)

In [ ]:
clean.groupby("FamilySize")["Survived"].agg(n="count", rate="mean").round(3)

## 步驟 5　畫圖

**🤖 給 AI 的 prompt**

> 請用 seaborn 畫三張圖，圖上文字用英文：
> 1. 性別、艙等各自的存活率長條圖，左右並排，y 軸 0 到 1。
> 2. 艙等 × 性別的存活率長條圖，不要誤差線，柱子上標出數值。
> 3. 存活與死亡乘客的年齡分布：依 Outcome 上色的堆疊直方圖，每 5 歲一格。

In [ ]:
sns.set_theme(style="whitegrid")                        # 白底加格線
fig, axes = plt.subplots(1, 2, figsize=(9, 4))          # 一列兩張
sns.barplot(data=clean, x="Sex", y="Survived", hue="Sex", order=["female", "male"],
            palette={"female": "#66c2a5", "male": "#fc8d62"}, legend=False, ax=axes[0])   # 長條 = 平均（存活率），黑線 = 95% 信賴區間
axes[0].set(title="Survival rate by sex", xlabel="Sex", ylabel="Survival rate", ylim=(0, 1))
sns.barplot(data=clean, x="Pclass", y="Survived", hue="Pclass",
            palette=["#1d3557", "#457b9d", "#8ecae6"], legend=False, ax=axes[1])
axes[1].set(title="Survival rate by class", xlabel="Passenger class", ylabel="Survival rate", ylim=(0, 1))
plt.tight_layout()
plt.show()

In [ ]:
ax = sns.barplot(data=clean, x="Pclass", y="Survived", hue="Sex", hue_order=["female", "male"],
                 palette={"female": "#66c2a5", "male": "#fc8d62"}, errorbar=None)          # 不畫誤差線
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.2f", padding=2)            # 柱子上標數值
ax.set(title="Survival rate by class and sex", xlabel="Passenger class",
       ylabel="Survival rate", ylim=(0, 1.08))
plt.show()

In [ ]:
ax = sns.histplot(data=clean, x="Age", hue="Outcome", multiple="stack",
                  binwidth=5, palette={"Died": "#9aa5b1", "Survived": "#2a9d8f"})
ax.set(title="Age of passengers (age known for 714)", xlabel="Age (years)", ylabel="Passengers")
plt.show()

## 步驟 6　統計檢定

**🤖 給 AI 的 prompt**

> 我想回答三個問題：
> 1. 性別和存活有沒有關係？
> 2. 艙等和存活有沒有關係？
> 3. 存活者和死亡者的年齡有沒有差別？
> 請先說明每個問題適合用哪一種統計檢定、為什麼，再寫程式。p 值用科學記號顯示。

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

tab = pd.crosstab(clean["Sex"], clean["Survived"])           # 2 × 2 列聯表
chi2, p, dof, expected = chi2_contingency(tab)
print(tab)
print(f"性別：chi2 = {chi2:.1f}, df = {dof}, p = {p:.2e}")

In [ ]:
tab2 = pd.crosstab(clean["Pclass"], clean["Survived"])       # 3 × 2 列聯表
chi2, p, dof, expected = chi2_contingency(tab2)
print(tab2)
print(f"艙等：chi2 = {chi2:.1f}, df = {dof}, p = {p:.2e}")

In [ ]:
age_s = clean.loc[clean["Survived"] == 1, "Age"].dropna()   # 存活者的年齡（去掉缺值）
age_d = clean.loc[clean["Survived"] == 0, "Age"].dropna()   # 死亡者的年齡
print("存活者：n =", len(age_s), " 中位數 =", age_s.median())
print("死亡者：n =", len(age_d), " 中位數 =", age_d.median())
stat, p = mannwhitneyu(age_s, age_d)
print(f"Mann-Whitney U = {stat:.0f}, p = {p:.3f}")

## 步驟 7　多變數模型：logistic regression

**🤖 給 AI 的 prompt**

> 性別、艙等、年齡彼此有關（例如頭等艙的乘客年紀比較大），分開檢定可能會互相混淆。
> 請用 statsmodels 做 logistic regression：Survived ~ 性別 + 艙等 + 年齡，
> 算出每個變數的 odds ratio、95% 信賴區間和 p 值，整理成一張表，並用白話解釋 odds ratio 的意思。

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

model = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age", data=clean).fit()   # Age 缺值的列會自動排除
print("用到的乘客數：", int(model.nobs))
or_table = pd.DataFrame({
    "OR":      np.exp(model.params),            # 係數取 e 次方 = odds ratio
    "CI_low":  np.exp(model.conf_int()[0]),
    "CI_high": np.exp(model.conf_int()[1]),
}).round(3)
or_table["p"] = model.pvalues.map(lambda v: f"{v:.1e}")   # p 值用科學記號，不然會被四捨五入成 0
or_table

**🤖 給 AI 的 prompt**

> 請把上面的 odds ratio 畫成森林圖：每個變數一列，點是 OR，橫線是 95% 信賴區間，
> x 軸用對數尺度，在 OR = 1 畫一條虛線，變數名稱改成好讀的英文。

In [ ]:
ors = or_table.drop("Intercept")                                  # 截距不畫
names = {"C(Sex)[T.male]": "Male (vs female)",
         "C(Pclass)[T.2]": "2nd class (vs 1st)",
         "C(Pclass)[T.3]": "3rd class (vs 1st)",
         "Age": "Age (per 1 year)"}
y = range(len(ors))
fig, ax = plt.subplots(figsize=(7, 3))
ax.errorbar(ors["OR"], y,
            xerr=[ors["OR"] - ors["CI_low"], ors["CI_high"] - ors["OR"]],
            fmt="o", color="#264653", capsize=4)
ax.axvline(1, linestyle="--", color="grey")                      # OR = 1：沒有關係
ax.set_xscale("log")
ax.set_yticks(list(y), [names[i] for i in ors.index])
ax.invert_yaxis()                                                # 第一個變數放最上面
ax.set(xlabel="Odds ratio of survival (95% CI, log scale)", title="Logistic regression")
plt.tight_layout()
plt.show()

## 步驟 8　結論

**🤖 給 AI 的 prompt**

> 以下是我所有的分析結果：
> （貼上：整體存活率、性別與艙等的存活率表、卡方檢定與 Mann–Whitney 的結果、logistic regression 的 OR 表）
> 請用繁體中文寫一段約 250 字的結論，給沒有統計背景的人看：
> 1. 只能使用我提供的數字，不要自己補數字。
> 2. 分成「主要發現」和「這份分析的限制」兩段。
> 3. 不要寫出「因為是女性所以活下來」這種因果的說法。

（把 AI 寫好、你逐句核對過數字的結論貼在這裡）